# What did the policy say on that date? Temporal search with Db2

One question, two customers: someone who booked in **March** and someone who booked in **June**
ask the same thing, and the right answer differs because the policy changed on 1 May. This notebook
stores dated policy clauses in a Db2 **bitemporal** table with a native `VECTOR` column, then answers
"which clause, in force on this date, best matches this claim?" in **one SQL statement**:
`FOR BUSINESS_TIME AS OF` picks the version, `VECTOR_DISTANCE` ranks by meaning.

Then it changes the policy with `UPDATE ... FOR PORTION OF BUSINESS_TIME` and replays the past with
`FOR SYSTEM_TIME AS OF`: what did the database hold *before* the change was recorded?

All data is fictional. Everything runs in its own schema, `TEMPORAL_RECIPE`, which the last cell drops.

## 0. Connect

Credentials come from `.env` (copy `.env.example`). Every value below is a bound parameter.

In [1]:
import os
import time

import ibm_db
import ibm_db_dbi
from dotenv import load_dotenv

load_dotenv()
dsn = (f"DATABASE={os.getenv('DB_NAME', 'SAMPLE')};HOSTNAME={os.getenv('DB_HOST', 'localhost')};"
       f"PORT={os.getenv('DB_PORT', '50000')};PROTOCOL={os.getenv('DB_PROTOCOL', 'tcpip')};")
conn = ibm_db_dbi.connect(dsn, os.getenv("DB_USER", "db2inst1"), os.environ["DB_PASSWORD"])
conn.set_autocommit(True)
cur = conn.cursor()
info = ibm_db.server_info(conn.conn_handler)
print(info.DBMS_NAME, info.DBMS_VER)  # native VECTOR needs 12.1.2 or later

DB2/LINUXX8664 12.01.0500


## 1. A table with two clocks

- **Business time** (`BUS_START`, `BUS_END`): when a clause version applies to customers.
- **System time** (`SYS_START`, `SYS_END`): when the database held that row. Db2 fills it in, and moves
  replaced rows to the history table by itself.

`PRIMARY KEY (CLAUSE_ID, BUSINESS_TIME WITHOUT OVERLAPS)` makes two versions of one clause in force on
the same day impossible.

In [2]:
SCHEMA = "TEMPORAL_RECIPE"

def run(sql, params=()):
    cur.execute(sql, params)

def try_run(sql):
    try:
        cur.execute(sql)
    except Exception:  # noqa: S110 - the object is not there yet: fine on a first run
        pass

run("SET CURRENT SCHEMA = ?", (SCHEMA,))
try_run("ALTER TABLE CLAUSE DROP VERSIONING")  # leftovers from an earlier run
try_run("DROP TABLE CLAUSE_HIST")
try_run("DROP TABLE CLAUSE")

run('''
CREATE TABLE CLAUSE (
  CLAUSE_ID   VARCHAR(16)   NOT NULL,
  CLAUSE_TEXT VARCHAR(1000) NOT NULL,
  EMBEDDING   VECTOR(384, FLOAT32),
  BUS_START   DATE          NOT NULL,
  BUS_END     DATE          NOT NULL,
  SYS_START   TIMESTAMP(12) NOT NULL GENERATED ALWAYS AS ROW BEGIN,
  SYS_END     TIMESTAMP(12) NOT NULL GENERATED ALWAYS AS ROW END,
  TRANS_ID    TIMESTAMP(12) GENERATED ALWAYS AS TRANSACTION START ID,
  PERIOD BUSINESS_TIME (BUS_START, BUS_END),
  PERIOD SYSTEM_TIME (SYS_START, SYS_END),
  PRIMARY KEY (CLAUSE_ID, BUSINESS_TIME WITHOUT OVERLAPS)
)''')
run("CREATE TABLE CLAUSE_HIST LIKE CLAUSE")
run("ALTER TABLE CLAUSE ADD VERSIONING USE HISTORY TABLE CLAUSE_HIST")
print("CLAUSE is bitemporal; replaced rows go to CLAUSE_HIST")

CLAUSE is bitemporal; replaced rows go to CLAUSE_HIST


## 2. Embed the clauses and load them

A small local model (`all-MiniLM-L6-v2`, 384 dimensions, CPU) embeds each clause. The vector travels as
a bound parameter in the text form Db2's `VECTOR(?, 384, FLOAT32)` accepts.

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

def vec(text):
    return "[" + ",".join(f"{x:.8g}" for x in model.encode(text, normalize_embeddings=True)) + "]"

OPEN = "9999-12-31"
clauses = [
    ("BRV-002", "Bereavement fares must be requested before travel. They cannot be applied retroactively.", "2026-01-01", OPEN),
    ("BAG-001", "The first checked bag costs $35 each way.", "2026-01-01", OPEN),
    ("PET-001", "Small cats and dogs may travel in the cabin for $100 each way.", "2026-01-01", OPEN),
]
for cid, text, start, end in clauses:
    run("INSERT INTO CLAUSE (CLAUSE_ID, CLAUSE_TEXT, EMBEDDING, BUS_START, BUS_END) "
        "VALUES (?, ?, VECTOR(?, 384, FLOAT32), ?, ?)", (cid, text, vec(text), start, end))
print(len(clauses), "clauses loaded")

3 clauses loaded


## 3. The policy changes on 1 May

`FOR PORTION OF BUSINESS_TIME` changes the clause **from 1 May onwards only**. Db2 splits the row in two:
the old text keeps January to April, the new text applies from May. The replaced row goes to history.

In [4]:
cur.execute("VALUES CURRENT TIMESTAMP")
before_change = cur.fetchone()[0]
time.sleep(1)  # so the replay below has a moment before the change to look at

new_text = "Bereavement fares may be requested retroactively within 90 days of the ticket being issued."
run("UPDATE CLAUSE FOR PORTION OF BUSINESS_TIME FROM ? TO ? "
    "SET CLAUSE_TEXT = ?, EMBEDDING = VECTOR(?, 384, FLOAT32) WHERE CLAUSE_ID = ?",
    ("2026-05-01", OPEN, new_text, vec(new_text), "BRV-002"))

cur.execute("SELECT CLAUSE_ID, BUS_START, BUS_END, LEFT(CLAUSE_TEXT, 60) FROM CLAUSE WHERE CLAUSE_ID = ? ORDER BY BUS_START",
            ("BRV-002",))
for row in cur.fetchall():
    print(row)

('BRV-002', datetime.date(2026, 1, 1), datetime.date(2026, 5, 1), 'Bereavement fares must be requested before travel. They cann')
('BRV-002', datetime.date(2026, 5, 1), datetime.date(9999, 12, 31), 'Bereavement fares may be requested retroactively within 90 d')


## 4. One query: in force on the date, ranked by meaning

The same claim, checked for a March booking and a June booking. Only the date parameter changes.

In [5]:
AS_OF = '''
SELECT c.CLAUSE_ID, c.BUS_START, c.CLAUSE_TEXT,
       VECTOR_DISTANCE(c.EMBEDDING, VECTOR(?, 384, FLOAT32), COSINE) AS DIST
FROM CLAUSE FOR BUSINESS_TIME AS OF ? AS c
ORDER BY DIST
FETCH FIRST 2 ROWS ONLY'''

claim = "You can claim the bereavement fare after you travel."
for day in ("2026-03-10", "2026-06-02"):
    cur.execute(AS_OF, (vec(claim), day))
    top = cur.fetchall()[0]
    print(f"{day}: {top[0]} from {top[1]} (distance {top[3]:.3f})\n    {top[2]}")

2026-03-10: BRV-002 from 2026-01-01 (distance 0.218)
    Bereavement fares must be requested before travel. They cannot be applied retroactively.
2026-06-02: BRV-002 from 2026-05-01 (distance 0.373)
    Bereavement fares may be requested retroactively within 90 days of the ticket being issued.


## 5. Replay: what did the database hold before the change?

`FOR SYSTEM_TIME AS OF` reads the table as it stood at an earlier moment. Asked for a June date, it
still returns the old text, because the May change had not been recorded yet. This is the evidence
you need when a complaint arrives months later: what the system knew when it answered.

In [6]:
REPLAY = '''
SELECT c.CLAUSE_ID, c.BUS_START, c.CLAUSE_TEXT
FROM CLAUSE FOR SYSTEM_TIME AS OF ? FOR BUSINESS_TIME AS OF ? AS c
WHERE c.CLAUSE_ID = ?'''

for label, moment in (("before the change", before_change), ("now", None)):
    if moment is None:
        cur.execute("SELECT CLAUSE_ID, BUS_START, CLAUSE_TEXT FROM CLAUSE FOR BUSINESS_TIME AS OF ? WHERE CLAUSE_ID = ?",
                    ("2026-06-02", "BRV-002"))
    else:
        cur.execute(REPLAY, (moment, "2026-06-02", "BRV-002"))
    row = cur.fetchone()
    print(f"2026-06-02, as the database held it {label}: {row[2]}")

cur.execute("SELECT COUNT(*) FROM CLAUSE_HIST")
print("rows in history:", cur.fetchone()[0])

2026-06-02, as the database held it before the change: Bereavement fares must be requested before travel. They cannot be applied retroactively.
2026-06-02, as the database held it now: Bereavement fares may be requested retroactively within 90 days of the ticket being issued.
rows in history: 1


## 6. Clean up

In [7]:
run("ALTER TABLE CLAUSE DROP VERSIONING")
run("DROP TABLE CLAUSE_HIST")
run("DROP TABLE CLAUSE")
try_run(f"DROP SCHEMA {SCHEMA} RESTRICT")  # a constant above, not input
conn.close()
print("dropped schema", SCHEMA)

dropped schema TEMPORAL_RECIPE


## What you just saw

- **Business time answers "what applied on that date".** `FOR BUSINESS_TIME AS OF ?` returned a different
  version for March and June from one statement, and `WITHOUT OVERLAPS` guarantees exactly one per date.
- **Temporal and vector predicates compose.** The as-of filter and `VECTOR_DISTANCE` ranking are one
  query, so there is no separate vector store to keep in step with the dated policy.
- **`FOR PORTION OF` does the bookkeeping.** One `UPDATE` split the row at 1 May; nobody computed periods by hand.
- **System time answers "what did we know then".** The replay read the table as it stood before the
  change, from rows Db2 moved to history by itself.